# LangChain Tools
**Tools** allow LLMs to take actions — search the web, run calculations, call APIs, or execute custom functions.

Instead of just generating text, the model can decide **which tool to call** and **with what inputs**.

In [23]:
!pip install langchain langchain-ollama --quiet

In [29]:
def simple_decorator(func):
    def wrapper():
        print("Before the function call")
        func()
        print("After the function call")
    return wrapper

@simple_decorator
def greet():
    print("Hello!")

greet()

Before the function call
Hello!
After the function call


In [28]:
func()

hello world


## 1. Create a Custom Tool
Use the `@tool` decorator to turn any Python function into a LangChain tool.

In [39]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    # Simulated response (replace with a real API call)
    return f'The weather in {city} is sunny and 25°C.'

# Inspect the tool
print('Tool name   :', add_numbers.name)
print('Description :', add_numbers.description)
print('Schema      :', add_numbers.args)

Tool name   : add_numbers
Description : Adds two numbers together.
Schema      : {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## 2. Call a Tool Directly

In [40]:
result = add_numbers.invoke({'a': 10, 'b': 25})
print('10 + 25 =', result)


weather = get_weather.invoke({'city': 'Paris'})
print(weather)

10 + 25 = 35
The weather in Paris is sunny and 25°C.


## 3. Bind Tools to the Model
When tools are bound, the model can decide to call them when needed.

In [42]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.2')
tools = [add_numbers, get_weather]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke('How is ther weather in paris')
print('Response   :', response.content)
print('Tool calls :', response.tool_calls)

Response   : 
Tool calls : [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'e464a7dd-e90c-4f8f-921d-970148448a5f', 'type': 'tool_call'}]


## 4. Execute the Tool Call the Model Requested

In [43]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {'add_numbers': add_numbers, 'get_weather': get_weather}

# Step 1 — ask the model
messages = [HumanMessage(content='What is the weather in Tokyo?')]
response = llm_with_tools.invoke(messages)

messages.append(response)

# Step 2 — run the tool the model asked for
for tool_call in response.tool_calls:
    tool_fn = tool_map[tool_call['name']]
    tool_result = tool_fn.invoke(tool_call['args'])
    messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call['id']))

print(messages)
# Step 3 — send result back to the model for a final answer
final = llm_with_tools.invoke(messages)
print('Final answer:', final.content)


[HumanMessage(content='What is the weather in Tokyo?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-03-08T16:04:39.24454Z', 'done': True, 'done_reason': 'stop', 'total_duration': 536066542, 'load_duration': 89988417, 'prompt_eval_count': 192, 'prompt_eval_duration': 144969750, 'eval_count': 18, 'eval_duration': 296443751, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019cce31-88af-7ea1-9b66-21c3242adb9c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': '636a71f8-c096-4f16-847e-275102551d0f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 18, 'total_tokens': 210}), ToolMessage(content='The weather in Tokyo is sunny and 25°C.', tool_call_id='636a71f8-c096-4f16-847e-275102551d0f')]
Final answer: It's a beautiful day in Tokyo! The current weather is mostly sunny with a tempe

## 5. Multiple Tools — Model Picks the Right One

In [22]:
questions = [
    'What is 42 + 58?',
    'What is the weather in London?'
]

for question in questions:
    response = llm_with_tools.invoke(question)
    if response.tool_calls:
        call = response.tool_calls[0]
        print(f'Q: {question}')
        print(f'   -> Model chose tool: {call["name"]} with args {call["args"]}')
    else:
        print(f'Q: {question}')
        print(f'   -> Direct answer: {response.content}')
    print()

Q: What is 42 + 58?
   -> Model chose tool: add_numbers with args {'a': '42', 'b': '58'}

Q: What is the weather in London?
   -> Model chose tool: get_weather with args {'city': 'London'}



## Summary

| Concept | Description |
|---|---|
| `@tool` | Decorator to turn a Python function into a LangChain tool |
| `tool.invoke()` | Call a tool directly with arguments |
| `llm.bind_tools()` | Give the model access to a list of tools |
| `response.tool_calls` | List of tools the model decided to call |
| `ToolMessage` | Pass tool result back to the model |
| Tool loop | Ask model → run tool → send result back → get final answer |